In [3]:
import sys

print(sys.executable)

C:\Users\dishu\anaconda3\python.exe


In [4]:
!{sys.executable} -m pip install mediapipe

  Using cached mediapipe-1.0.1-py3-none-win_amd64.whl.metadata (10 kB)
Using cached mediapipe-1.0.1-py3-none-win_amd64.whl (20.1 MB)


In [3]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)

MediaPipe version: 1.0.1


In [4]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("MediaPipe Tasks API loaded successfully!")

MediaPipe Tasks API loaded successfully!


In [6]:
model_path = "hand_landmarker.task"

print("Model path set successfully!")

Model path set successfully!


In [7]:
base_options = python.BaseOptions(
    model_asset_path=model_path
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

print("Hand Landmarker options configured!")

Hand Landmarker options configured!


In [8]:
hand_landmarker = vision.HandLandmarker.create_from_options(options)

print("Hand Landmarker created successfully!")

Hand Landmarker created successfully!


In [9]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam")
else:
    print("Webcam opened successfully!")

Webcam opened successfully!


In [10]:
ret, frame = cap.read()

if ret:
    print("Frame captured successfully!")
    print("Frame shape:", frame.shape)
else:
    print("Failed to capture frame")

Frame captured successfully!
Frame shape: (480, 640, 3)


In [11]:
rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

print("Frame converted from BGR to RGB!")

Frame converted from BGR to RGB!


In [12]:
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb_frame
)

print("MediaPipe image created successfully!")

MediaPipe image created successfully!


In [13]:
timestamp_ms = int(time.time() * 1000)

result = hand_landmarker.detect_for_video(
    mp_image,
    timestamp_ms
)

print("Hand detection completed!")
print("Number of hands detected:", len(result.hand_landmarks))

Hand detection completed!
Number of hands detected: 0


In [14]:
if result.hand_landmarks:
    hand = result.hand_landmarks[0]

    index_tip = hand[8]

    print("Index finger tip:")
    print("x:", index_tip.x)
    print("y:", index_tip.y)
    print("z:", index_tip.z)
else:
    print("No hand detected")

No hand detected


In [15]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if result.hand_landmarks:
        print("Hand detected")
        break

    cv2.imshow("Hand Detection Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [16]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Draw landmarks
        for landmark in hand:
            x = int(landmark.x * frame.shape[1])
            y = int(landmark.y * frame.shape[0])

            cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

        # Draw connections
        connections = mp.tasks.vision.HandLandmarksConnections.HAND_CONNECTIONS

        for connection in connections:
            start = hand[connection.start]
            end = hand[connection.end]

            start_point = (
                int(start.x * frame.shape[1]),
                int(start.y * frame.shape[0])
            )

            end_point = (
                int(end.x * frame.shape[1]),
                int(end.y * frame.shape[0])
            )

            cv2.line(
                frame,
                start_point,
                end_point,
                (255, 0, 0),
                2
            )

    cv2.imshow("Hand Landmarks", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [17]:
keys = [
    ["Q", "W", "E", "R", "T", "Y", "U", "I", "O", "P"],
    ["A", "S", "D", "F", "G", "H", "J", "K", "L"],
    ["Z", "X", "C", "V", "B", "N", "M"],
    ["SPACE", "BACKSPACE", "ENTER"]
]

print("Keyboard layout created!")

Keyboard layout created!


In [18]:
key_width = 55
key_height = 55

start_x = 0
start_y = 200

key_gap = 8

print("Keyboard dimensions updated!")

Keyboard dimensions updated!


In [19]:
def draw_keyboard(frame, keys, start_x, start_y, key_width, key_height, key_gap):

    key_positions = []

    frame_width = frame.shape[1]

    for row_index, row in enumerate(keys):

        # Give special keys larger widths
        widths = []

        for key in row:
            if key == "SPACE":
                widths.append(150)
            elif key == "BACKSPACE":
                widths.append(120)
            elif key == "ENTER":
                widths.append(90)
            else:
                widths.append(key_width)

        # Calculate total width of current row
        row_width = sum(widths) + (len(row) - 1) * key_gap

        # Center the row
        row_x = (frame_width - row_width) // 2

        current_x = row_x

        for col_index, key in enumerate(row):

            current_width = widths[col_index]

            x = current_x
            y = start_y + row_index * (key_height + key_gap)

            # Draw key
            cv2.rectangle(
                frame,
                (x, y),
                (x + current_width, y + key_height),
                (255, 255, 255),
                2
            )

            # Text size
            font_scale = 0.55

            text_size = cv2.getTextSize(
                key,
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                2
            )[0]

            # Center text inside key
            text_x = x + (current_width - text_size[0]) // 2
            text_y = y + (key_height + text_size[1]) // 2

            cv2.putText(
                frame,
                key,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                (255, 255, 255),
                2
            )

            # Store key boundaries
            key_positions.append(
                (
                    key,
                    x,
                    y,
                    x + current_width,
                    y + key_height
                )
            )

            current_x += current_width + key_gap

    return key_positions

In [20]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    cv2.imshow("Virtual Keyboard", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [21]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # Draw keyboard
    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    # Detect index finger
    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        index_tip = hand[8]

        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Draw index fingertip
        cv2.circle(
            frame,
            (finger_x, finger_y),
            10,
            (0, 255, 0),
            -1
        )

    cv2.imshow("Virtual Keyboard", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [22]:
def get_key_at_position(finger_x, finger_y, key_positions):

    for key, x1, y1, x2, y2 in key_positions:

        if x1 <= finger_x <= x2 and y1 <= finger_y <= y2:
            return key

    return None

In [23]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    # Convert BGR → RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Create MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Timestamp
    timestamp_ms = int(time.time() * 1000)

    # Detect hand
    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # Draw keyboard
    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    # Detect index finger
    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        index_tip = hand[8]

        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Find the key under the finger
        selected_key = get_key_at_position(
            finger_x,
            finger_y,
            key_positions
        )

        # Draw fingertip
        cv2.circle(
            frame,
            (finger_x, finger_y),
            10,
            (0, 255, 0),
            -1
        )

        # Show selected key
        if selected_key:
            cv2.putText(
                frame,
                f"Selected: {selected_key}",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )

    cv2.imshow("Virtual Keyboard", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [31]:
def is_pinching(hand, threshold=0.08):

    thumb_tip = hand[4]
    index_tip = hand[8]

    distance = math.sqrt(
        (thumb_tip.x - index_tip.x) ** 2 +
        (thumb_tip.y - index_tip.y) ** 2
    )

    return distance < threshold

In [25]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        index_tip = hand[8]
        thumb_tip = hand[4]

        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Check pinch
        pinching = is_pinching(hand)

        # Draw index finger
        cv2.circle(
            frame,
            (finger_x, finger_y),
            10,
            (0, 255, 0),
            -1
        )

        if pinching:
            cv2.putText(
                frame,
                "PINCH DETECTED",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )
        else:
            cv2.putText(
                frame,
                "Pointing",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2
            )

    cv2.imshow("Pinch Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [33]:
typed_text = ""

last_key = None
last_press_time = 0

key_cooldown = 0.2

pinch_active = False

print("Typing variables initialized!")

Typing variables initialized!


In [27]:
def process_key(key):
    global typed_text

    if key == "SPACE":
        typed_text += " "

    elif key == "BACKSPACE":
        typed_text = typed_text[:-1]

    elif key == "ENTER":
        typed_text += "\n"

    else:
        typed_text += key

In [28]:
def draw_keyboard(
    frame,
    keys,
    start_x,
    start_y,
    key_width,
    key_height,
    key_gap,
    selected_key=None,
    pressed_key=None
):
    
    key_positions = []
    frame_width = frame.shape[1]

    for row_index, row in enumerate(keys):

        widths = []

        for key in row:
            if key == "SPACE":
                widths.append(150)
            elif key == "BACKSPACE":
                widths.append(120)
            elif key == "ENTER":
                widths.append(90)
            else:
                widths.append(key_width)

        row_width = sum(widths) + (len(row) - 1) * key_gap
        row_x = (frame_width - row_width) // 2

        current_x = row_x

        for col_index, key in enumerate(row):

            current_width = widths[col_index]

            x = current_x
            y = start_y + row_index * (key_height + key_gap)

            # Select appearance
            if key == pressed_key:
                thickness = -1
            elif key == selected_key:
                thickness = 3
            else:
                thickness = 2

            cv2.rectangle(
                frame,
                (x, y),
                (x + current_width, y + key_height),
                (255, 255, 255),
                thickness
            )

            font_scale = 0.55

            text_size = cv2.getTextSize(
                key,
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                2
            )[0]

            text_x = x + (current_width - text_size[0]) // 2
            text_y = y + (key_height + text_size[1]) // 2

            cv2.putText(
                frame,
                key,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                (255, 255, 255),
                2
            )

            key_positions.append(
                (
                    key,
                    x,
                    y,
                    x + current_width,
                    y + key_height
                )
            )

            current_x += current_width + key_gap

    return key_positions

In [35]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    # Draw keyboard
    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    # Convert BGR → RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    # Detect hand
    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    selected_key = None
    pressed_key = None

    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        # Index fingertip
        index_tip = hand[8]

        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Find key under finger
        selected_key = get_key_at_position(
            finger_x,
            finger_y,
            key_positions
        )

        # Check pinch
        pinching = is_pinching(hand)

       # Draw a clearly visible fingertip marker
        cv2.circle(
            frame,
            (finger_x, finger_y),
            15,
            (0, 255, 0),
            -1
        )
        
        cv2.circle(
            frame,
            (finger_x, finger_y),
            18,
            (255, 255, 255),
            2
        )

        # Press only when pinch STARTS
        if pinching and not pinch_active:

            if selected_key is not None:

                process_key(selected_key)

                last_key = selected_key
                pressed_key = selected_key

                print("Pressed:", selected_key)

            pinch_active = True

        # Reset after fingers separate
        if not pinching:
            pinch_active = False

    else:
        # No hand → reset pinch state
        pinch_active = False

    # Display typed text
    cv2.rectangle(
        frame,
        (20, 20),
        (frame.shape[1] - 20, 90),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        "Text: " + typed_text[-40:],
        (30, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.imshow("Virtual Keyboard", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [38]:
pinch_frames = 0
required_pinch_frames = 3

typed_text = ""

print("Key press system initialized!")

Key press system initialized!


In [40]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    # Detect hand
    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # Draw keyboard
    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    selected_key = None

    # If hand detected
    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        # Index finger tip = landmark 8
        index_tip = hand[8]

        # Convert normalized coordinates to pixels
        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Find key
        selected_key = get_key_at_position(
            finger_x,
            finger_y,
            key_positions
        )

        # GREEN DOT
        cv2.circle(
            frame,
            (finger_x, finger_y),
            15,
            (0, 255, 0),
            -1
        )

        # Display coordinates
        cv2.putText(
            frame,
            f"X: {finger_x}  Y: {finger_y}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Display selected key
        if selected_key is not None:

            cv2.putText(
                frame,
                "KEY: " + selected_key,
                (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

    else:

        cv2.putText(
            frame,
            "NO HAND",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    cv2.imshow("Virtual Keyboard Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [41]:
pinch_was_active = False
last_pressed_key = None

print("Key press control initialized!")

Key press control initialized!


In [42]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to capture frame")
        break

    frame = cv2.flip(frame, 1)

    # Convert BGR → RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int(time.time() * 1000)

    # Detect hand
    result = hand_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # Draw keyboard
    key_positions = draw_keyboard(
        frame,
        keys,
        start_x,
        start_y,
        key_width,
        key_height,
        key_gap
    )

    selected_key = None
    pressed_key = None

    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        # Index fingertip
        index_tip = hand[8]

        finger_x = int(index_tip.x * frame.shape[1])
        finger_y = int(index_tip.y * frame.shape[0])

        # Find key under finger
        selected_key = get_key_at_position(
            finger_x,
            finger_y,
            key_positions
        )

        # Detect pinch
        pinching = is_pinching(hand)

        # Press only when pinch STARTS
        if pinching and not pinch_was_active:

            if selected_key is not None:

                process_key(selected_key)

                last_pressed_key = selected_key
                pressed_key = selected_key

                print("Pressed:", selected_key)

            pinch_was_active = True

        # Reset after releasing pinch
        elif not pinching:

            pinch_was_active = False
            last_pressed_key = None

        # Draw fingertip
        cv2.circle(
            frame,
            (finger_x, finger_y),
            15,
            (0, 255, 0),
            -1
        )

        # Show selected key
        if selected_key is not None:

            cv2.putText(
                frame,
                "KEY: " + selected_key,
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        # Show pinch status
        if pinching:

            cv2.putText(
                frame,
                "PINCH",
                (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

    else:

        # Reset if hand disappears
        pinch_was_active = False

    # Display typed text
    cv2.rectangle(
        frame,
        (20, 90),
        (frame.shape[1] - 20, 145),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        "Text: " + typed_text[-40:],
        (30, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.imshow("Virtual Keyboard", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Pressed: D
Pressed: D
Pressed: P
Pressed: L
Pressed: L
Pressed: B
Pressed: SPACE
Pressed: SPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: C
Pressed: L
Pressed: N
Pressed: B
Pressed: B
Pressed: BACKSPACE
Pressed: V
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: SPACE
Pressed: D
Pressed: T
Pressed: SPACE
Pressed: B
Pressed: H
Pressed: H
Pressed: H
Pressed: SPACE
Pressed: SPACE
Pressed: SPACE
Pressed: SPACE
Pressed: SPACE
Pressed: V
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: BACKSPACE
Pressed: SPACE
Pressed: BACKSPACE
Pressed: V
Pressed: BACKSPACE
Pressed: V


In [43]:
def is_pinching(hand, threshold=0.06):

    thumb_tip = hand[4]
    index_tip = hand[8]

    distance = math.sqrt(
        (thumb_tip.x - index_tip.x) ** 2 +
        (thumb_tip.y - index_tip.y) ** 2 +
        (thumb_tip.z - index_tip.z) ** 2
    )

    return distance < threshold